# Demographic summary tables

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Summarize included and excluded participants and the two HC clusters in discovery and validation cohorts.


## Inputs

- `files/tables/full_risk_factors_[1, 2].xlsx`
- `files/tables/full_risk_factors_[3].xlsx`

These full-cohort workbooks require CODE, IDX, Diagnosis, and the listed demographic fields. Their export is inactive in the supplied risk-factor notebook; provide existing workbooks before running.

## Outputs

- `files/tables/Demographics_Table.xlsx`: included participants and HC clusters.
- `files/tables/Demographics_Excluded_Table.xlsx`: excluded participants.



## 1. Paths and cohort-specific variables

Use the specified continuous and categorical variable lists and display-name mapping.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "tables"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

cont_var_disc = [
    'Age',
    'Total amount of children',
    'Weight (gramm)',
    'Stressful life events'
]

categ_var_disc = [
    'Highest degree of education',
    'Income',
    'Baby blues',
    'Marital status',
    'Breastfeeding T0',
    'Breastfeeding T4',
    'Birth mode',
    'Relocation to other ward',
    'Gender of the child',
    'Psychiatric diagnosis in previous pregancy',
    'Familial psychiatric history',
    'Birth-related psychological and physical traumas',
    'Psychiatric history binary',
    'Psychiatric history',
    'Diagnosis'
]

cont_var_val = [
    'Age',
    'Total amount of children',
    'Height (cm)',
    'BMI before pregnancy',
    'BMI after birth',
    'BMI 3 months after birth',
    'Weight (gramm)',
    'Stressful life events',
    'RLS_Wert',
    'PMS_value',
    'Ablehnung T1',
    'Ablehnung T2',
    'Ablehnung T3',
    'Ablehnung T4',
    'Severity of baby blues',
    'Empathy',
    'System',
    'Mutter Care',
    'Mutter Overprotection',
    'Vater Care',
    'Vater Overprotection'
]

categ_var_val = [
    'Highest degree of education',
    'Income',
    'Support at home t0',
    'Support at home t4',
    'PMS Severity',
    'Baby blues',
    'Marital status',
    'Breastfeeding T0',
    'Breastfeeding T4',
    'in vitro fertilization2',
    'Birth mode',
    'Complications at birth',
    'Complication during pregancy',
    'Relocation to other ward',
    'Gender of the child',
    'Psychiatric diagnosis in previous pregancy',
    'Familial psychiatric history',
    'RLS',
    'Birth-related psychological and physical traumas',
    'Psychiatric history binary',
    'Psychiatric history',
    'Diagnosis'
]

rename_dict_coh = {
    "Highest degree of education": "Education",
    "Income": "Annual income, thousand €",
    "Baby blues": "Baby Blues",
    "Marital status": "Married",
    "Breastfeeding T0": "Breastfeeding at childbirth",
    "Breastfeeding T4": "Breastfeeding at 12 weeks pp",
    "Gender of the child": "Gender of the baby",
    "Birth-related psychological and physical traumas":
        "Subjective experience of traumatic childbirth",
    "Support at home t0": "Support at home at childbirth",
    "Support at home t4": "Support at home at 12 weeks pp",
    "Psychiatric history binary": "Psychiatric history",
    "Age": "Age, years",
    "Total amount of children": "Total number of children",
    "Stressful life events": "Stressful life events (number)",
    "Weight (gramm)": "Birthweight of the child, g",
    "Height (cm)": "Height, cm",
    "Ablehnung T4": "Rejection towards the baby T4",
    "Ablehnung T1": "Rejection towards the baby T1",
    "Ablehnung T2": "Rejection towards the baby T2",
    "Ablehnung T3": "Rejection towards the baby T3",
    "Mutter Care": "Care from own mother",
    "Vater Care": "Care from own father",
    "Mutter Overprotection": "Overprotection by own mother",
    "Vater Overprotection": "Overprotection by own father",
    "RLS_Wert": "RLS score",
    "PMS_value": "PMS score"
}


## 2. Format available observations

Continuous summaries report mean ± sample SD (ddof=1) and N after numeric coercion and missing-value removal. Categorical summaries report counts and percentages of nonmissing observations, with N. Empty groups are blank; a single continuous observation has undefined SD.

In [ ]:
def format_continuous(series):
    """Format available numeric scores as mean ± sample SD and N."""

    series = pd.to_numeric(series, errors='coerce').dropna()

    n = len(series)

    if n == 0:
        return ""

    mean = series.mean()
    sd = series.std()

    return f"{mean:.2f} ± {sd:.2f} (N={n})"

def format_categorical(series):
    """Format category counts and nonmissing-denominator percentages."""

    series = series.dropna()

    n = len(series)

    if n == 0:
        return ""

    counts = series.value_counts()

    output = []

    for category, count in counts.items():
        percentage = count / n * 100
        output.append(f"{category}: {count} ({percentage:.1f}%)")

    return f"{'; '.join(output)} (N={n})"


## 3. Define participant groups

Included means IDX is nonmissing; excluded means IDX is missing. HC clusters require CODE=ND and IDX=0 (Cluster 1) or IDX=1 (Cluster 2). These are descriptive summaries, without significance tests.

In [ ]:
def create_cohort_table(df, cont_vars, categ_vars, cohort_name):

    """Summarize included, excluded, and HC cluster groups for one cohort."""
    rows = []

    included = df[df["IDX"].notna()].copy()
    excluded = df[df["IDX"].isna()].copy()

    hc_c1 = df[(df["CODE"] == "ND") & (df["IDX"] == 0)].copy()
    hc_c2 = df[(df["CODE"] == "ND") & (df["IDX"] == 1)].copy()

    for var in cont_vars:

        row = {
            "Variable": rename_dict_coh.get(var, var),
            f"{cohort_name} included": format_continuous(included[var]),
            f"{cohort_name} excluded": format_continuous(excluded[var]),
            f"{cohort_name} HC Cluster 1": format_continuous(hc_c1[var]),
            f"{cohort_name} HC Cluster 2": format_continuous(hc_c2[var])
        }

        rows.append(row)

    for var in categ_vars:

        row = {
            "Variable": rename_dict_coh.get(var, var),
            f"{cohort_name} included": format_categorical(included[var]),
            f"{cohort_name} excluded": format_categorical(excluded[var]),
            f"{cohort_name} HC Cluster 1": format_categorical(hc_c1[var]),
            f"{cohort_name} HC Cluster 2": format_categorical(hc_c2[var])
        }

        rows.append(row)

    return pd.DataFrame(rows)


## 4. Build both cohort tables

In [ ]:
discovery = pd.read_excel(
    INPUT_PATH / r'full_risk_factors_[1, 2].xlsx'
)

discovery_table = create_cohort_table(
    discovery,
    cont_var_disc,
    categ_var_disc,
    "Discovery"
)

validation = pd.read_excel(
    INPUT_PATH / r'full_risk_factors_[3].xlsx'
)

validation_table = create_cohort_table(
    validation,
    cont_var_val,
    categ_var_val,
    "Validation"
)


## 5. Merge and export

Outer-join by the displayed variable name. Both psychiatric-history fields share the display name “Psychiatric history”; the existing join can therefore create duplicate combinations for that label. This behavior is preserved.

In [ ]:
main_table = discovery_table.merge(
    validation_table,
    on="Variable",
    how="outer"
)

main_table = main_table[
    [
        "Variable",
        "Discovery included",
        "Discovery HC Cluster 1",
        "Discovery HC Cluster 2",
        "Validation included",
        "Validation HC Cluster 1",
        "Validation HC Cluster 2"
    ]
]

print(main_table.to_string(index=False))

excluded_table = discovery_table[[
    "Variable",
    "Discovery excluded"
]].merge(
    validation_table[[
        "Variable",
        "Validation excluded"
    ]],
    on="Variable",
    how="outer"
)

print(excluded_table.to_string(index=False))

main_output = INPUT_PATH / r'Demographics_Table.xlsx'
excluded_output = INPUT_PATH / r'Demographics_Excluded_Table.xlsx'

main_table.to_excel(
    main_output,
    index=False
)

excluded_table.to_excel(
    excluded_output,
    index=False
)

print(f"\nMain table saved to:\n{main_output}")
print(f"Excluded participants table saved to:\n{excluded_output}")


In [ ]:
discovery_table
